In [19]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import signal
from scipy.ndimage import gaussian_filter1d
import anndata as ad
from tqdm import tqdm
from scipy.signal import savgol_filter, find_peaks
from pybaselines import whittaker


# === Core Functions ===



def create_global_spectrum(X, mz_axis):
    if hasattr(X, "toarray"):
        X = X.toarray()
    global_spectrum = np.mean(X, axis=0)  # use average instead of sum
    return mz_axis, global_spectrum

import numpy as np
from scipy.signal import savgol_filter, find_peaks
from pybaselines import whittaker


def detect_peaks_hdi_style(
    mz_axis: np.ndarray,
    intensity: np.ndarray,
    smooth_window: int = 11,
    smooth_polyorder: int = 3,
    baseline_lambda: float = 1e5,
    baseline_p: float = 0.01,
    prominence: float = 5000,
    min_distance: float = 0.015,
    top_n: int = 150
):
    """
    Detect peaks in a global spectrum in a style similar to Waters HDImaging.

    Parameters:
        mz_axis (np.ndarray): 1D array of m/z values.
        intensity (np.ndarray): 1D array of intensities (same shape as mz_axis).
        smooth_window (int): Window length for Savitzky-Golay smoothing.
        smooth_polyorder (int): Polynomial order for smoothing.
        baseline_lambda (float): λ parameter for ALS baseline correction.
        baseline_p (float): p parameter for ALS.
        prominence (float): Minimum prominence for peak detection.
        min_distance (float): Minimum m/z distance between peaks.
        top_n (int): Number of top peaks to return based on intensity.

    Returns:
        List of (mz, intensity) tuples for detected peaks.
    """

    # Step 1: Smoothing
    smoothed = savgol_filter(intensity, smooth_window, smooth_polyorder)

    # Step 2: Baseline correction (ALS)
    corrected, _ = whittaker.asls(smoothed, lam=baseline_lambda, p=baseline_p)


    # Step 3: Peak detection based on prominence
    peak_indices, properties = find_peaks(corrected, prominence=prominence)
    peak_mzs = mz_axis[peak_indices]
    peak_intensities = corrected[peak_indices]

    # Step 4: Sort by intensity
    sorted_peaks = sorted(zip(peak_mzs, peak_intensities), key=lambda x: x[1], reverse=True)

    # Step 5: Remove peaks that are too close to each other
    filtered_peaks = []
    for mz, inten in sorted_peaks:
        if all(abs(mz - fmz) > min_distance for fmz, _ in filtered_peaks):
            filtered_peaks.append((mz, inten))
        if len(filtered_peaks) >= top_n:
            break

    return filtered_peaks

In [ ]:
# === Parameters ===
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a.h5ad"
# === Load and Process ===

adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

intensity_matrix = adata.X  # This could be sparse


global_mz = adata.var["mzs"].values
global_intensity = intensity_matrix.sum(axis=0).A1  # if sparse

peaks = detect_peaks_hdi_style(global_mz, global_intensity)

for mz, inten in peaks:
    print(f"{mz:.4f}\t{inten:.2f}")

/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning:

Transforming to str index.



Loaded AnnData from: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a.h5ad
271.2750	100314.55
782.4820	62885.45
796.2950	56552.76
1007.2396	54290.76
810.5024	52789.57
758.3174	49667.18
830.4250	48116.63
806.4551	47964.30
897.1548	46116.63
948.6557	46027.75
775.0858	44379.97
635.1874	44161.15
814.4850	43712.21
852.4370	43581.26
552.1720	43371.86
1025.0129	42226.30
1079.0078	41532.91
964.6180	41128.60
1020.9910	41103.43
727.1309	40820.36
874.4400	40701.28
1045.0723	40380.47
1166.7188	40378.26
925.2008	40371.14
820.4648	40302.03
1057.0619	40260.20
1143.0226	39776.90
919.2606	39598.76
546.1658	39585.24
1066.9510	39474.96
756.4189	39129.20
1188.7754	39007.78
991.0506	38628.68
1095.0590	38614.83
902.2232	38346.94
792.4064	38147.85
1357.1306	37801.61
844.3994	37569.70
836.3492	37390.82
881.2576	37291.12
790.5270	37192.87
1135.0062	36799.19
846.4180	36793.57
661.1534	36524.37
869.3860	36492.37
1118.4595	36295.47
1335.4067	36115.91
838.4720	35846.33
1204.9532	35611.09
1295.6895	35555.46
848.37

: 